# E1〜E6 / T1〜T3の時系列CV比較 + ROC-AUC Ensemble

保存済みEmbedding、表形式特徴、char TF-IDFを既存のexpanding-window foldsで比較します。前処理、TF-IDF、PCAは各foldのtrainingだけでfitします。初期状態では学習しません。

T4 x1を利用する既定設定です。E3/E4はPyTorch CUDA + mixed precision、E5はCatBoost GPU、E6はXGBoost CUDAを使います。

In [ ]:
from pathlib import Path
import logging
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ensemble import (
    evaluate_fold_auc,
    hill_climb_auc,
    make_profile_submissions,
    save_ensemble_outputs,
)
from embedding_features import load_embeddings
from modeling import (
    default_modeling_config,
    fit_full_and_predict_test,
    run_all_experiments,
)
from validation import encode_binary_target, make_time_series_cv

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

## 1. データと特徴列

列名は実データに合わせて編集してください。`project_start_year == -1`は既存CV関数がfoldから除外し、表特徴側でもmissingとして扱います。

In [ ]:
TARGET_COL = 'science_tech_decision'
YEAR_COL = 'project_start_year'
PROJECT_COL = 'project_name'
ID_COL = 'project_id'
TEXT_COLS = ['project_name', 'project_objective', 'project_summary']

NUMERIC_COLS = [
    'project_start_year',
    'project_end_year',
    'project_fiscal_year',
    'budget',
]
CATEGORICAL_COLS = [
    'responsible_ministry',
]

train = pd.read_csv(PROJECT_ROOT / 'input' / 'train.csv')
test = pd.read_csv(PROJECT_ROOT / 'input' / 'test.csv')
train[TARGET_COL] = encode_binary_target(train[TARGET_COL])  # 該当=1, 非該当=0
print('train:', train.shape, 'test:', test.shape)

## 2. Embedding cacheを明示的に選択

Bedrock model ID・region・adapterごとのcache切り替えは`EMBEDDING_CACHE_DIR`だけで行います。候補を表示してから、train/testの両方が入った同じdirectoryを指定してください。行順はmetadataの`project_id`と元indexで検証されます。

In [ ]:
EMBEDDING_ROOT = PROJECT_ROOT / 'data' / 'embeddings'
available_caches = sorted(path for path in EMBEDDING_ROOT.glob('*') if path.is_dir())
for path in available_caches:
    print(path)

# 例: EMBEDDING_CACHE_DIR = available_caches[0]
EMBEDDING_CACHE_DIR = None

In [ ]:
train_embeddings = test_embeddings = None
train_embedding_metadata = test_embedding_metadata = None
if EMBEDDING_CACHE_DIR is not None:
    train_embeddings, train_embedding_metadata = load_embeddings(
        EMBEDDING_CACHE_DIR,
        split='train',
        expected_df=train,
        project_id_col=ID_COL,
    )
    test_embeddings, test_embedding_metadata = load_embeddings(
        EMBEDDING_CACHE_DIR,
        split='test',
        expected_df=test,
        project_id_col=ID_COL,
    )
    print('train embedding:', train_embeddings.shape)
    print('test embedding :', test_embeddings.shape)
else:
    print('EMBEDDING_CACHE_DIRを選択してください。')

## 3. 全実験で共有する時系列fold

In [ ]:
folds, cv_diagnostics = make_time_series_cv(
    df=train,
    year_col=YEAR_COL,
    project_col=PROJECT_COL,
    target_col=TARGET_COL,
    n_valid_years=3,
)
display(cv_diagnostics)

## 4. T4向けconfig

通常のE6はPCAなしです。`[None, 512, 256]`へ変えた場合だけ追加PCA実験を行います。CatBoost GPUは演算順の都合でbitwise deterministicではありません。

In [ ]:
CONFIG = default_modeling_config()
CONFIG['output_dir'] = str(PROJECT_ROOT / 'outputs')
CONFIG['tfidf_feature_output_dir'] = str(PROJECT_ROOT / 'data' / 'csv' / 'tfidf_shared')
CONFIG['text_cols'] = TEXT_COLS
CONFIG['metric'] = 'roc_auc'  # このコンペの評価指標
CONFIG['e6_pca_dims'] = [None]

# T4 x1
CONFIG['mlp']['device'] = 'cuda'
CONFIG['mlp']['batch_size'] = 256
CONFIG['mlp']['use_amp'] = True
CONFIG['mlp']['early_stop_metric'] = 'auc'  # 学習lossはBCE、best epochはAUCで選択
CONFIG['catboost']['task_type'] = 'GPU'
CONFIG['catboost']['devices'] = '0'
CONFIG['catboost']['eval_metric'] = 'AUC'  # loss_functionはLoglossのまま
CONFIG['xgboost']['device'] = 'cuda'
CONFIG['xgboost']['tree_method'] = 'hist'
CONFIG['xgboost']['eval_metric'] = 'auc'

# 最初はCVだけ。test予測はCV結果を見た後に下の専用セルで実行する。
CONFIG['run_final_test_prediction'] = False
CONFIG

In [ ]:
try:
    import torch
    print('torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
except ImportError:
    print('torch is not installed. Run: python -m pip install -r requirements.txt')

## 5. E1〜E6 / T1〜T3を実行

`RUN_CV=True`にした場合だけ学習します。すべて同じ`folds`を受け取ります。T1〜T3は各foldで一度だけ作ったTF-IDFを共有し、OOFは`outputs/oof_predictions.parquet`へ統一保存されます。

In [ ]:
RUN_CV = False
suite = None
if RUN_CV:
    if train_embeddings is None:
        raise RuntimeError('先にEMBEDDING_CACHE_DIRを選択してください。')
    suite = run_all_experiments(
        train=train,
        folds=folds,
        train_embeddings=train_embeddings,
        train_embedding_metadata=train_embedding_metadata,
        numeric_cols=NUMERIC_COLS,
        categorical_cols=CATEGORICAL_COLS,
        config=CONFIG,
        target_col=TARGET_COL,
        project_col=PROJECT_COL,
        project_id_col=ID_COL,
        year_col=YEAR_COL,
    )
    display(suite.summary.sort_values('mean', ascending=CONFIG['metric'] == 'log_loss'))
else:
    print('CV is disabled. Set RUN_CV=True when ready.')

## 6. fold・seen/unseen・時間・PCAの詳細

In [ ]:
if suite is not None:
    detail_cols = [
        'experiment', 'validation_year', 'overall_score',
        'seen_score', 'unseen_score', 'seen_ratio',
        'fit_seconds', 'predict_seconds', 'input_dim',
        'device', 'best_iteration', 'early_stop_metric',
        'best_validation_auc', 'best_validation_loss', 'pca_explained_variance',
        'n_nonzero', 'sparse_memory_mib', 'shared_tfidf_seconds',
    ]
    available_detail_cols = [column for column in detail_cols if column in suite.fold_metrics]
    display(suite.fold_metrics[available_detail_cols])
    display(suite.oof_predictions.notna().sum().rename('OOF rows').to_frame())

## Optional: E6のPCA比較

必要なときだけ`CONFIG['e6_pca_dims'] = [None, 512, 256]`へ変更してCVセルを再実行します。各foldのPCAはfold training embeddingだけでfitされ、explained varianceがfold metricsへ保存されます。

## 7. OOF ROC-AUCでhill climbing ensemble

評価指標はROC-AUCです。年度driftに対する仮説を`ENSEMBLE_PROFILES`へ複数定義し、fold重みとprobability/rank blendを比較します。rankはモデルごと・foldごとにpercentile rank化します。比較結果から最高値を自動採用するとOOFへ過適合しやすいため、基準となる設定は`SELECTED_ENSEMBLE_PROFILE`で明示します。

In [ ]:
RUN_ENSEMBLE = False
ENSEMBLE_CANDIDATES = None  # Noneなら実行した全モデル。必要ならモデル名listを指定。
SELECTED_ENSEMBLE_PROFILE = 'recent_20_30_50_rank'  # outputs/submission.csvにも保存する基準profile
ENSEMBLE_PROFILES = {
    'uniform_rank': {
        'objective': 'weighted_fold_auc', 'blend_mode': 'rank', 'fold_weights': [1, 1, 1],
    },
    'recent_20_30_50_rank': {
        'objective': 'weighted_fold_auc', 'blend_mode': 'rank', 'fold_weights': [0.2, 0.3, 0.5],
    },
    'recent_10_20_70_rank': {
        'objective': 'weighted_fold_auc', 'blend_mode': 'rank', 'fold_weights': [0.1, 0.2, 0.7],
    },
    'recent_20_30_50_probability': {
        'objective': 'weighted_fold_auc', 'blend_mode': 'probability', 'fold_weights': [0.2, 0.3, 0.5],
    },
}
ensemble_results = {}
ensemble_result = None

if RUN_ENSEMBLE:
    if suite is None:
        raise RuntimeError('先にRUN_CV=TrueでOOFを作成してください。')
    if CONFIG['metric'] != 'roc_auc':
        raise RuntimeError('hill climbingの目的関数はこのコンペのROC-AUCです。')
    for name, settings in ENSEMBLE_PROFILES.items():
        if len(settings['fold_weights']) != len(folds):
            raise RuntimeError(f'{name}のfold_weightsを実際のfold数に合わせてください。')
    comparison_rows = []
    for name, settings in ENSEMBLE_PROFILES.items():
        result = hill_climb_auc(
            oof_predictions=suite.oof_predictions,
            target=train[TARGET_COL],
            candidate_models=ENSEMBLE_CANDIDATES,
            folds=folds,
            max_steps=50,
            weight_grid=np.arange(0.05, 0.55, 0.05),
            min_improvement=1e-6,
            **settings,
        )
        ensemble_results[name] = result
        fold_auc = evaluate_fold_auc(
            result.oof_prediction, train[TARGET_COL], folds, years=train[YEAR_COL]
        )
        comparison_rows.append({
            'name': name,
            'objective': result.objective,
            'blend_mode': result.blend_mode,
            'objective_score': result.score,
            'pooled_auc': result.pooled_auc,
            'mean_fold_auc': fold_auc['roc_auc'].mean(),
            'latest_fold_auc': fold_auc.iloc[-1]['roc_auc'],
            'n_models': int(result.weights.gt(0).sum()),
            'fold_weights': str(settings['fold_weights']),
        })
        save_ensemble_outputs(result, PROJECT_ROOT / 'outputs' / 'ensembles' / name)
    ensemble_comparison = pd.DataFrame(comparison_rows).set_index('name')
    ensemble_comparison.to_csv(PROJECT_ROOT / 'outputs' / 'ensemble_profile_comparison.csv')
    display(ensemble_comparison)
    if SELECTED_ENSEMBLE_PROFILE not in ensemble_results:
        raise KeyError(f'Unknown SELECTED_ENSEMBLE_PROFILE: {SELECTED_ENSEMBLE_PROFILE}')
    ensemble_result = ensemble_results[SELECTED_ENSEMBLE_PROFILE]
    print('selected:', SELECTED_ENSEMBLE_PROFILE)
    print('objective score:', ensemble_result.score)
    print('pooled OOF AUC:', ensemble_result.pooled_auc)
    print('scored OOF rows:', ensemble_result.n_scored_rows)
    display(ensemble_result.individual_scores.to_frame())
    display(ensemble_result.weights[ensemble_result.weights > 0].sort_values(ascending=False).to_frame())
    display(ensemble_result.history)
    selected_fold_auc = evaluate_fold_auc(
        ensemble_result.oof_prediction,
        train[TARGET_COL],
        folds,
        years=train[YEAR_COL],
    )
    display(selected_fold_auc)
    print('mean fold AUC:', selected_fold_auc['roc_auc'].mean())
    print('latest fold AUC:', selected_fold_auc.iloc[-1]['roc_auc'])
else:
    print('Ensemble is disabled. Set RUN_ENSEMBLE=True after CV.')

## 8. 選抜モデルを全trainで再fitし、複数submissionを作成

全profileで正のensemble weightを持つモデルの和集合だけを一度ずつ再学習し、test予測を共有してprofile別CSVを作ります。`ID_COL`はtestの値と行順をそのまま保持します。基準profileは従来互換の`outputs/submission.csv`にも保存します。`RUN_FINAL_SUBMISSION=True`へ変更するまで実行されません。

In [ ]:
RUN_FINAL_SUBMISSION = False
SUBMISSION_PATH = PROJECT_ROOT / 'outputs' / 'submission.csv'
SUBMISSION_DIR = PROJECT_ROOT / 'outputs' / 'submissions'

if RUN_FINAL_SUBMISSION:
    if not ensemble_results:
        raise RuntimeError('先にRUN_ENSEMBLE=Trueでensemble profileを作成してください。')
    final_experiments = sorted({
        experiment
        for result in ensemble_results.values()
        for experiment in result.weights[result.weights > 0].index
    })
    embedding_free = {'E5_tabular_catboost', 'T1_tfidf_lr', 'T2_tfidf_tabular_lr'}
    if any(name not in embedding_free for name in final_experiments) and (train_embeddings is None or test_embeddings is None):
        raise RuntimeError('選択した実験にはtrain/test Embeddingが必要です。')
    prediction_dir = PROJECT_ROOT / 'outputs' / 'test_predictions'
    prediction_dir.mkdir(parents=True, exist_ok=True)
    test_predictions = {}
    for experiment in final_experiments:
        pca_dim = suite.results[experiment].metadata.get('pca_dim')
        prediction = fit_full_and_predict_test(
            experiment=experiment,
            train=train,
            test=test,
            train_embeddings=train_embeddings,
            test_embeddings=test_embeddings,
            train_embedding_metadata=train_embedding_metadata,
            test_embedding_metadata=test_embedding_metadata,
            numeric_cols=NUMERIC_COLS,
            categorical_cols=CATEGORICAL_COLS,
            config=CONFIG,
            pca_dim=pca_dim,
            target_col=TARGET_COL,
            project_id_col=ID_COL,
        )
        test_predictions[experiment] = prediction
        np.save(prediction_dir / f'{experiment}.npy', prediction.astype(np.float32))
        print(experiment, prediction.shape, prediction.min(), prediction.max())

    submissions, submission_manifest = make_profile_submissions(
        test,
        test_predictions,
        ensemble_results,
        id_col=ID_COL,
        prediction_col=TARGET_COL,
        output_dir=SUBMISSION_DIR,
        canonical_profile=SELECTED_ENSEMBLE_PROFILE,
        canonical_output_path=SUBMISSION_PATH,
    )
    reloaded = pd.read_csv(SUBMISSION_PATH)
    assert reloaded.columns.tolist() == [ID_COL, TARGET_COL]
    assert reloaded[ID_COL].astype(str).tolist() == test[ID_COL].astype(str).tolist()
    assert len(reloaded) == len(test)
    assert len(submissions) == len(ENSEMBLE_PROFILES)
    display(submission_manifest)
    display(submissions[SELECTED_ENSEMBLE_PROFILE].head())
    print('saved canonical:', SUBMISSION_PATH, 'rows:', len(reloaded))
    print('saved variants:', SUBMISSION_DIR)
else:
    print('Final submission is disabled. Set RUN_FINAL_SUBMISSION=True after ensemble selection.')